> **AI-Assisted Runbook — Academic Integrity Disclosure**
>
> This notebook was developed with the assistance of **GitHub Copilot**, an AI pair-programming tool. The initial code structure and implementation were AI-generated. However, I have personally reviewed every cell, understood the logic, debugged issues, and made corrections as needed. The final, working solution represents my own understanding and effort.
>
> This disclosure is made in accordance with **Udacity's plagiarism and academic integrity policy**, which requires transparency when AI tools are used as part of the learning process.

# Udaplay - Part 1: Vector Database & Data Ingestion

This notebook loads the local dataset of video game information (a personalized dataset covering
35 games across retro consoles, modern consoles, PC, mobile, and VR), cleans and formats it into
RAG-ready documents, embeds those documents, and stores them in a **persistent ChromaDB vector
database** so they can later be retrieved with semantic search.

**Steps covered:**
1. Load the raw JSON array from `games_dataset.json`
2. Clean and format each record into a single natural-language document + metadata
3. Create a persistent Chroma collection with a sentence-embedding function
4. Add all documents (with embeddings + metadata) to the collection
5. Query the collection with natural-language questions to demonstrate semantic search


In [1]:
import json
import os
import glob

import chromadb
from chromadb.utils import embedding_functions


## 1. Load the raw dataset

All games are provided as a single JSON array in `games_dataset.json`.

In [2]:
from pathlib import Path

GAMES_FILE = "games_dataset.json"

if not Path(GAMES_FILE).is_file():
    matches = list(Path(".").rglob(GAMES_FILE))
    if not matches:
        raise FileNotFoundError(
            f"Could not find {GAMES_FILE!r}. Place it in the notebook folder or update GAMES_FILE."
        )
    GAMES_FILE = str(matches[0])

with open(GAMES_FILE, "r", encoding="utf-8") as f:
    raw_games = json.load(f)

print(f"Loaded {len(raw_games)} game records")
raw_games[0]


Loaded 35 game records


{'Name': 'Gran Turismo',
 'Platform': 'PlayStation 1',
 'Genre': 'Racing',
 'Publisher': 'Sony Computer Entertainment',
 'YearOfRelease': 1997,
 'Rating': 87,
 'Tags': ['racing', 'simulation', 'cars', 'single-player'],
 'Description': 'A realistic racing simulator featuring a large roster of licensed cars, detailed physics, and a career mode that lets players earn licenses and upgrade vehicles.',
 'Review': 'Astonishingly realistic for its time, though the licensing tests can feel like a slog.'}

## 2. Clean and format the data

For each game we:
- normalize/validate the expected fields (`Name`, `Platform`, `Genre`, `Publisher`, `YearOfRelease`, `Description`)
- build a single natural-language **document string** that reads well for embedding and retrieval
- keep the structured fields as **metadata** so we can filter/inspect results later


In [3]:
def clean_text(value):
    """Basic text cleanup: strip whitespace, collapse internal spaces."""
    if value is None:
        return ""
    return " ".join(str(value).split())


def format_game_document(game):
    """Turn a structured game record into one RAG-friendly text chunk."""
    name = clean_text(game.get("Name"))
    platform = clean_text(game.get("Platform"))
    genre = clean_text(game.get("Genre"))
    publisher = clean_text(game.get("Publisher"))
    year = clean_text(game.get("YearOfRelease"))
    description = clean_text(game.get("Description"))
    tags = ", ".join(game.get("Tags", []))
    review = clean_text(game.get("Review"))

    return (
        f"{name} is a {genre} game released in {year} for {platform}, "
        f"published by {publisher}. {description} "
        f"Tags: {tags}. Player review: {review}"
    )


documents = []      # the text chunks that get embedded
metadatas = []       # structured metadata per chunk
ids = []             # unique id per chunk

for i, game in enumerate(raw_games):
    doc_text = format_game_document(game)
    documents.append(doc_text)
    metadatas.append({
        "Name": clean_text(game.get("Name")),
        "Platform": clean_text(game.get("Platform")),
        "Genre": clean_text(game.get("Genre")),
        "Publisher": clean_text(game.get("Publisher")),
        "YearOfRelease": game.get("YearOfRelease"),
        "Rating": game.get("Rating"),
        "Tags": ", ".join(game.get("Tags", [])),
    })
    ids.append(f"game_{i:03d}")

print(documents[0])


Gran Turismo is a Racing game released in 1997 for PlayStation 1, published by Sony Computer Entertainment. A realistic racing simulator featuring a large roster of licensed cars, detailed physics, and a career mode that lets players earn licenses and upgrade vehicles. Tags: racing, simulation, cars, single-player. Player review: Astonishingly realistic for its time, though the licensing tests can feel like a slog.


## 3. Create a persistent Chroma vector database

We use `chromadb.PersistentClient` so the collection is written to disk (`./chroma_db`) and
survives across notebook restarts, rather than living only in memory.

We use OpenAI's `text-embedding-3-small` model (via `OpenAIEmbeddingFunction`) to turn each
document into a dense embedding vector. This only needs a lightweight HTTPS call per batch of
text (no multi-hundred-MB model download), so it works well even in restricted network
environments.

**API key handling:** the key is never hard-coded in this notebook. It's read from the
`OPENAI_API_KEY` environment variable if set, otherwise you'll be prompted to enter it securely
(hidden input, not echoed or stored in the notebook file).


In [4]:
import os
import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]


In [5]:
CHROMA_PATH = "chroma_db"
COLLECTION_NAME = "video_games"

client = chromadb.PersistentClient(path=CHROMA_PATH)

embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=OPENAI_API_KEY,
    model_name="text-embedding-3-small",
)

# get_or_create so re-running the notebook doesn't error on a duplicate collection
collection = client.get_or_create_collection(
    name=COLLECTION_NAME,
    embedding_function=embedding_fn,
    metadata={"hnsw:space": "cosine"},
)

print("Collection ready:", collection.name)


Collection ready: video_games


## 4. Add the processed documents to the vector database

In [6]:
# Upsert so this cell is safe to re-run without creating duplicate entries
collection.upsert(
    ids=ids,
    documents=documents,
    metadatas=metadatas,
)

print(f"Collection now contains {collection.count()} documents")


Collection now contains 35 documents


## 5. Query the vector database with semantic search

These queries use natural language rather than exact keywords, showing that retrieval is based on
**meaning**, not just string matching.


In [7]:
def semantic_search(query, n_results=3):
    results = collection.query(query_texts=[query], n_results=n_results)
    print(f"Query: {query!r}\n")
    for rank, (doc, meta, dist) in enumerate(
        zip(results["documents"][0], results["metadatas"][0], results["distances"][0]), start=1
    ):
        print(f"{rank}. {meta['Name']} ({meta['Platform']}, {meta['YearOfRelease']}) "
              f"- distance={dist:.4f}")
        print(f"   {doc}\n")


semantic_search("an open world cowboy game")


Query: 'an open world cowboy game'

1. Red Dead Redemption 2 (Xbox One, 2018) - distance=0.5026
   Red Dead Redemption 2 is a Action-Adventure game released in 2018 for Xbox One, published by Rockstar Games. Outlaw Arthur Morgan navigates the decline of the Wild West as his gang faces internal betrayal and encroaching civilization in a richly detailed frontier. Tags: open-world, western, story-rich, single-player. Player review: Astonishingly detailed, though the deliberate pacing won't be for everyone.

2. Grand Theft Auto V (PlayStation 3, 2013) - distance=0.5861
   Grand Theft Auto V is a Action-Adventure game released in 2013 for PlayStation 3, published by Rockstar Games. An open-world crime saga set in Los Santos following three protagonists whose stories intertwine through heists, satire, and sprawling exploration. Tags: open-world, crime, multiplayer, satire. Player review: A staggering open world packed with detail, even if the tone can feel mean-spirited at times.

3. Guild W

In [8]:
semantic_search("racing games for PlayStation")


Query: 'racing games for PlayStation'

1. Gran Turismo (PlayStation 1, 1997) - distance=0.3947
   Gran Turismo is a Racing game released in 1997 for PlayStation 1, published by Sony Computer Entertainment. A realistic racing simulator featuring a large roster of licensed cars, detailed physics, and a career mode that lets players earn licenses and upgrade vehicles. Tags: racing, simulation, cars, single-player. Player review: Astonishingly realistic for its time, though the licensing tests can feel like a slog.

2. FIFA 23 (PlayStation 5, 2022) - distance=0.6210
   FIFA 23 is a Sports game released in 2022 for PlayStation 5, published by Electronic Arts. The final entry under the FIFA license simulates professional soccer with licensed leagues, HyperMotion animation capture, and cross-play modes. Tags: sports, multiplayer, soccer, simulation. Player review: Solid on-pitch gameplay but plagued by familiar franchise mode complaints.

3. Mario Kart 8 Deluxe (Nintendo Switch, 2017) - dista

In [9]:
semantic_search("a Nintendo platformer with Mario")


Query: 'a Nintendo platformer with Mario'

1. Super Mario World (Super Nintendo Entertainment System, 1990) - distance=0.3809
   Super Mario World is a Platformer game released in 1990 for Super Nintendo Entertainment System, published by Nintendo. A side-scrolling platformer starring Mario and Yoshi as they traverse Dinosaur Land to rescue Princess Toadstool from Bowser. Tags: platformer, family-friendly, classic, single-player. Player review: Tight, joyful platforming that still holds up decades later.

2. Mario Kart 8 Deluxe (Nintendo Switch, 2017) - distance=0.5448
   Mario Kart 8 Deluxe is a Racing game released in 2017 for Nintendo Switch, published by Nintendo. An enhanced kart racer with anti-gravity tracks, a large roster of Nintendo characters, and robust local and online multiplayer modes. Tags: racing, multiplayer, family-friendly, party. Player review: The best couch multiplayer racer around, chaotic in all the right ways.

3. Sonic the Hedgehog (Sega Genesis, 1991) - dist

In [10]:
semantic_search("post-apocalyptic survival story with strong characters")


Query: 'post-apocalyptic survival story with strong characters'

1. The Last of Us (PlayStation 3, 2013) - distance=0.6143
   The Last of Us is a Action-Adventure game released in 2013 for PlayStation 3, published by Sony Computer Entertainment. Joel and Ellie navigate a fungal-infection apocalypse across the ruins of the United States, blending stealth combat with an emotionally driven narrative. Tags: story-rich, survival, single-player, post-apocalyptic. Player review: One of gaming's most affecting stories, carried by phenomenal performances.

2. Fallout 4 (Xbox One, 2015) - distance=0.6374
   Fallout 4 is a Action RPG game released in 2015 for Xbox One, published by Bethesda Softworks. A lone survivor wakes decades after a nuclear war to explore the ruins of Boston, building settlements and shaping the wasteland's factions. Tags: open-world, post-apocalyptic, single-player, rpg. Player review: A sprawling post-apocalyptic playground let down by a middling main story.

3. Red Dead 

### Richer queries over the personalized dataset

The queries below specifically probe the newly added companies, platforms, and genres
(mobile, VR, MOBA, MMORPG, retro consoles) to show the collection generalizes beyond the
original 20 games.


In [11]:
semantic_search("relaxing mobile puzzle game I can play on the bus")


Query: 'relaxing mobile puzzle game I can play on the bus'

1. Candy Crush Saga (Mobile (iOS / Android), 2012) - distance=0.5425
   Candy Crush Saga is a Puzzle game released in 2012 for Mobile (iOS / Android), published by King. A match-three puzzle game with thousands of levels, boosters, and social features that made it one of the most downloaded mobile games ever. Tags: puzzle, mobile, casual, free-to-play. Player review: Simple, addictive puzzle fun, though the monetization can feel aggressive.

2. Beat Saber (VR (Meta Quest), 2018) - distance=0.6944
   Beat Saber is a Rhythm game released in 2018 for VR (Meta Quest), published by Beat Games. Players slice incoming blocks in time with music using motion-tracked lightsabers, blending rhythm-game precision with full-body movement. Tags: vr, rhythm, music, single-player, fitness. Player review: One of the best VR showcases around, exhilarating and easy to pick up.

3. Portal 2 (PC, 2011) - distance=0.7010
   Portal 2 is a Puzzle game

In [12]:
semantic_search("VR rhythm game where you swing lightsabers to music")


Query: 'VR rhythm game where you swing lightsabers to music'

1. Beat Saber (VR (Meta Quest), 2018) - distance=0.3600
   Beat Saber is a Rhythm game released in 2018 for VR (Meta Quest), published by Beat Games. Players slice incoming blocks in time with music using motion-tracked lightsabers, blending rhythm-game precision with full-body movement. Tags: vr, rhythm, music, single-player, fitness. Player review: One of the best VR showcases around, exhilarating and easy to pick up.

2. Elden Ring (PlayStation 5, 2022) - distance=0.6735
   Elden Ring is a Action RPG game released in 2022 for PlayStation 5, published by Bandai Namco Entertainment. An open-world dark fantasy epic co-created with George R. R. Martin, challenging players with punishing combat and a mysterious, sprawling Lands Between. Tags: open-world, fantasy, difficult, single-player. Player review: Breathtaking scale and freedom, but brutally unforgiving for newcomers to the genre.

3. Dark Souls (Xbox 360, 2011) - distan

In [13]:
semantic_search("competitive online game with a big esports scene")


Query: 'competitive online game with a big esports scene'

1. League of Legends (PC, 2009) - distance=0.5358
   League of Legends is a MOBA game released in 2009 for PC, published by Riot Games. Two teams of five champions battle across three lanes to destroy the enemy Nexus, becoming one of the most-watched esports in the world. Tags: moba, competitive, multiplayer, esports, free-to-play. Player review: A deep strategic MOBA that remains fun, though the toxic community can be discouraging.

2. Guild Wars 2 (PC, 2012) - distance=0.6455
   Guild Wars 2 is a MMORPG game released in 2012 for PC, published by ArenaNet. An MMORPG built around dynamic world events and large-scale WvW battles, notable for dropping traditional questing hubs and trinity-based combat. Tags: mmorpg, multiplayer, fantasy, open-world. Player review: A refreshing MMO with dynamic events, though the story pacing can be uneven.

3. Overwatch (PC, 2016) - distance=0.6515
   Overwatch is a First-Person Shooter game rele

In [14]:
semantic_search("gritty stealth game from the late 90s with cinematic cutscenes")


Query: 'gritty stealth game from the late 90s with cinematic cutscenes'

1. Metal Gear Solid (PlayStation 1, 1998) - distance=0.4586
   Metal Gear Solid is a Stealth Action game released in 1998 for PlayStation 1, published by Konami. Solid Snake infiltrates a nuclear weapons facility to stop a rogue special forces unit, pioneering cinematic cutscenes and tactical stealth gameplay. Tags: stealth, story-rich, single-player, military. Player review: A cinematic masterpiece that reinvented stealth gameplay with unforgettable characters.

2. Hollow Knight (PC / Nintendo Switch / Steam Deck, 2017) - distance=0.5977
   Hollow Knight is a Metroidvania game released in 2017 for PC / Nintendo Switch / Steam Deck, published by Team Cherry. Players explore the ruined insect kingdom of Hallownest, gaining new abilities to reach previously inaccessible areas in this hand-drawn adventure. Tags: metroidvania, indie, atmospheric, single-player, difficult. Player review: A gorgeous, atmospheric metroid

In [15]:
semantic_search("cozy farming game to unwind after work")


Query: 'cozy farming game to unwind after work'

1. Stardew Valley (PC / Nintendo Switch / Mobile, 2016) - distance=0.4379
   Stardew Valley is a Farming Sim game released in 2016 for PC / Nintendo Switch / Mobile, published by ConcernedApe. Players inherit a rundown farm and grow crops, raise animals, and build relationships with townsfolk in a cozy pixel-art life sim. Tags: farming, relaxing, indie, single-player, family-friendly. Player review: Endless charm and an addictive farming loop, though grinding can feel repetitive late-game.

2. Animal Crossing: New Horizons (Nintendo Switch, 2020) - distance=0.5767
   Animal Crossing: New Horizons is a Life Simulation game released in 2020 for Nintendo Switch, published by Nintendo. Players cultivate a deserted island into a thriving community, decorating, fishing, and befriending anthropomorphic villagers at their own pace. Tags: life-sim, relaxing, family-friendly, multiplayer. Player review: A cozy, low-pressure escape, arriving right 

### Metadata filtering + semantic search combined

Chroma also supports filtering by metadata (`where=`) alongside the semantic query, letting us
narrow results to a specific publisher, platform, or minimum rating.


In [16]:
# Only highly-rated (>=90) games, ranked semantically by the query
results = collection.query(
    query_texts=["an emotional story-driven adventure"],
    n_results=5,
    where={"Rating": {"$gte": 90}},
)
for meta in results["metadatas"][0]:
    print(f"{meta['Name']} ({meta['Platform']}) - Rating {meta['Rating']}")


The Last of Us (PlayStation 3) - Rating 95
Hollow Knight (PC / Nintendo Switch / Steam Deck) - Rating 90
The Legend of Zelda: Ocarina of Time (Nintendo 64) - Rating 99
Red Dead Redemption 2 (Xbox One) - Rating 97
Elden Ring (PlayStation 5) - Rating 96


In [17]:
# Only games published by Nintendo
results = collection.query(
    query_texts=["fun game for the whole family"],
    n_results=5,
    where={"Publisher": "Nintendo"},
)
for meta in results["metadatas"][0]:
    print(f"{meta['Name']} ({meta['Platform']}) - {meta['Genre']}")


Super Mario World (Super Nintendo Entertainment System) - Platformer
Mario Kart 8 Deluxe (Nintendo Switch) - Racing
Animal Crossing: New Horizons (Nintendo Switch) - Life Simulation
The Legend of Zelda: Ocarina of Time (Nintendo 64) - Action-Adventure


## Summary

- Loaded a **personalized** local video game dataset (35 games, 27 publishers, spanning retro
  consoles, modern consoles, PC, mobile, and VR) from `games_dataset.json`
- Cleaned and formatted each record into a RAG-ready document + metadata (including new `Rating`
  and `Tags` fields)
- Embedded the documents with OpenAI's `text-embedding-3-small` model and stored them in a
  **persistent** ChromaDB collection (`./chroma_db`)
- Demonstrated **semantic search** across the expanded catalog, including queries that only make
  sense with the new data (mobile, VR, MOBA, retro consoles)
- Demonstrated **combining semantic search with metadata filters** (`where=`) for rating and
  publisher-scoped queries

This vector database is now ready to be used as the retrieval component of a RAG pipeline
(Part 2: building the agent that queries it and generates answers).